In [92]:
import os
import certifi
import requests

from dotenv import load_dotenv, find_dotenv
from langsmith import Client
from langchain_google_genai import GoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_agent
from langchain_classic.agents import AgentExecutor, create_react_agent
from langchain.tools import tool


In [93]:
_= load_dotenv(find_dotenv())
os.environ["SSL_CERT_FILE"] = certifi.where()
gemini_api_key = os.environ["GEMINI_API_KEY"]
tavily_api_key = os.environ["TAVILY_API_KEY"]
gemini_model = os.environ["GEMINI_MODEL"]
weather_api_key = os.environ["WEATHER_API_KEY"]


In [94]:
import time

EPA_AQI_LABELS = {
    "1": "Good",
    "2": "Moderate",
    "3": "Unhealthy for Sensitive Groups",
    "4": "Unhealthy",
    "5": "Very Unhealthy",
    "6": "Hazardous",
}


@tool
def get_weather(city: str) -> str:
    """Get the current weather and air quality (AQI, PM2.5, PM10) for a given city."""
    url = "http://api.weatherstack.com/current"
    params = {"access_key": weather_api_key, "query": city.strip()}

    response = requests.get(url, params=params)
    if response.status_code == 429:  # rate limited: wait and retry once
        time.sleep(2)
        response = requests.get(url, params=params)
    data = response.json()

    # Weatherstack can return errors with HTTP 200, so check the body instead
    if "current" not in data:
        error = data.get("error", {}).get("info", "unknown error")
        return f"Could not fetch weather data for {city}: {error}"

    current = data["current"]
    result = (
        f"City: {city}\n"
        f"Temperature: {current['temperature']}°C\n"
        f"Weather: {current['weather_descriptions'][0]}\n"
        f"Humidity: {current['humidity']}%"
    )

    aq = current.get("air_quality")
    if aq:
        epa_index = aq.get("us-epa-index")
        label = EPA_AQI_LABELS.get(epa_index, "Unknown")
        result += (
            f"\nAir Quality (US EPA index): {epa_index} - {label}\n"
            f"PM2.5: {aq.get('pm2_5')} µg/m³\n"
            f"PM10: {aq.get('pm10')} µg/m³"
        )

    return result

In [95]:
search_tool = TavilySearchResults(api_key=tavily_api_key,max_results=2)

In [96]:
result = search_tool.invoke("What is LangChain")

In [97]:
result

[{'title': 'LangChain - Wikipedia',
  'url': 'https://en.wikipedia.org/wiki/LangChain',
  'content': "Free and open-source software portal\n\nLangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain's use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n## History\n\n[edit] [...] ## Capabilities\n\n[edit]\n\nLangChain's developers highlight the framework's applicability to use-cases including chatbots, retrieval-augmented generation, document summarization, and synthetic data generation. InfoWorld described LangChain as a software development kit that simplifies the connection between large language models and external applications through a unified API. The magazine also wrote that it can be used to bring in context from sources such as PDFs, web pages, CSV files and relation

In [98]:
llm = GoogleGenerativeAI(api_key=gemini_api_key, model=gemini_model, temperature=0)

In [99]:
prompt = PromptTemplate.from_template("""Answer the following questions as best you can. You have access to the following tools:

{tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {input}
Thought:{agent_scratchpad}""")


In [100]:
tools = [search_tool, get_weather]

In [101]:
agents = create_react_agent(llm=llm, tools=tools, prompt=prompt)

In [102]:
agent_executor = AgentExecutor(agent=agents, tools=tools, verbose=True, handle_parsing_errors=True)

In [109]:
response = agent_executor.invoke({"input":  (
        "Find the capital of Australia"
        "and then find its current weather and AQI."
    )})



> Entering new AgentExecutor chain...


/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Question: Find the capital of Australia and then find its current weather and AQI.
Thought: I need to find the capital of Australia first, and then get the weather and AQI for that city.
Action: tavily_search_results_json
Action Input: what is the capital of Australia[{'title': 'Canberra - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Canberra', 'content': 'Canberra (/ˈkænbrə/ ⓘ KAN-brə; Ngunawal: Kanbarra) is the capital city of Australia and the largest population centre in the Australian Capital Territory. With an estimated population of 484,630 as of 2025, Canberra is Australia\'s largest inland city and the eighth-largest Australian city by population. The city occupies most of the northern quadrant of the Australian Capital Territory, and its urban sprawl borders the state of New South Wales to the northwest, northeast and southeast. The Brindabellas are located south and west of the city, and form the northern part of the Australian Alps, the country\'s highest mountain rang

/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


देखें, ऑस्ट्रेलिया की राजधानी **कैनबरा (Canberra)** है। अब मैं कैनबरा का वर्तमान मौसम और AQI (वायु गुणवत्ता सूचकांक) पता करता हूँ।

Action: get_weather
Action Input: CanberraCould not fetch weather data for Canberra: You have not supplied an API Access Key. [Required format: access_key=YOUR_ACCESS_KEY]

/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Thought: मुझे वेदर एपीआई की कुंजियाँ उपलब्ध नहीं हैं, इसलिए मैं वेब सर्च का उपयोग करके कैनबरा का वर्तमान मौसम और AQI ढूंढने का प्रयास करता हूँ।
Action: tavily_search_results_json
Action Input: Canberra current weather and AQI[{'title': 'Canberra Air Quality Forecast - Today, Tomorrow', 'url': 'https://airmine.ai/air-quality/australia/canberra', 'content': 'On 11 November 2026, air quality conditions in Canberra will be similar to today, with AQI forecast at good at 19.6/100. Pollution peaks around 7:00 PM (NO2:', 'score': 0.6784335}, {'title': 'Canberra Air Quality Index (AQI) | Air Pollution', 'url': 'https://www.aqi.in/us/dashboard/australia/australian-capital-territory/canberra', 'content': 'Current Canberra Air Quality Index (AQI) is 21 Good level with real-time air pollution PM2.5. Temperature (19.2°C) Updated: 2026-11-06', 'score': 0.6136997}]

/home/kashif/Desktop/learning/fde/agentic-ai/langchain-single-agents/.venv/lib/python3.11/site-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'models/gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Final Answer: ऑस्ट्रेलिया की राजधानी **कैनबरा (Canberra)** है। कैनबरा का वर्तमान मौसम और वायु गुणवत्ता (AQI):
- **AQI:** लगभग 21 (Good level - अच्छी स्थिति), जिसमें PM2.5 शामिल है।
- **तापमान:** लगभग 19.2°C।

> Finished chain.


In [110]:
response["output"]

'ऑस्ट्रेलिया की राजधानी **कैनबरा (Canberra)** है। कैनबरा का वर्तमान मौसम और वायु गुणवत्ता (AQI):\n- **AQI:** लगभग 21 (Good level - अच्छी स्थिति), जिसमें PM2.5 शामिल है।\n- **तापमान:** लगभग 19.2°C।'